# Задача 2: Titanic (Kaggle) — EDA, модели sklearn/XGBoost/CatBoost, SHAP

**Цель:** предсказать выживание пассажира (`Survived`: 0 — нет, 1 — да).

**План:**
1. Загрузка датасета (аналог соревнования Kaggle *Titanic - Machine Learning from Disaster*).
2. Разведочный анализ (EDA) с графиками.
3. Подготовка признаков и обучение моделей.
4. Сравнение метрик и разбор важности признаков через **SHAP**.

Каждый блок кода сопровождается комментариями: что делает импорт и зачем выполняется шаг.

## 0. Установка зависимостей (при необходимости)

Если пакетов нет в окружении, раскомментируйте и выполните ячейку **один раз** в Jupyter:

In [ ]:
# %pip install pandas numpy matplotlib seaborn scikit-learn xgboost catboost shap

## 1. Импорты

Ниже — стандартные библиотеки для анализа данных, визуализации, машинного обучения и интерпретации моделей.

In [ ]:
# --- Работа с данными и числами ---
import warnings  # Позволяет отключать лишние предупреждения, чтобы вывод был чище
from pathlib import Path  # Удобные пути к файлам (кроссплатформенно)

import numpy as np  # Массивы и быстрые численные операции
import pandas as pd  # Таблицы (DataFrame), чтение CSV, группировки

# --- Визуализация ---
import matplotlib.pyplot as plt  # Базовые графики (histogram, bar, ...)
import seaborn as sns  # Статистические графики поверх matplotlib, красивые темы

# --- Машинное обучение (scikit-learn) ---
from sklearn.model_selection import train_test_split  # Разбиение на train/test
from sklearn.compose import ColumnTransformer  # Разная обработка числовых и категориальных столбцов
from sklearn.pipeline import Pipeline  # Объединение препроцессинга + модели в один объект
from sklearn.impute import SimpleImputer  # Заполнение пропусков (медиана, мода, ...)
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # Кодирование категорий и масштабирование
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

# Модели (линейная классификация, деревья, ансамбли, KNN)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# Градиентный бустинг — отдельные библиотеки (часто используют вместе со sklearn-Pipeline)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# SHAP — объяснение вклада признаков в предсказание для каждого объекта
import shap

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

# Путь к CSV: скачан с открытого зеркала того же датасета, что на Kaggle
DATA_PATH = Path("titanic.csv")
assert DATA_PATH.exists(), (
    "Файл titanic.csv не найден. Положите его рядом с ноутбуком "
    "или скачайте train.csv с Kaggle и переименуйте."
)

print("Готово: библиотеки загружены, файл данных:", DATA_PATH.resolve())

## 2. Загрузка и первичный осмотр данных

Датасет Titanic содержит демографию пассажиров и факт выживания. На Kaggle обычно два файла (`train.csv`, `test.csv`); для обучения с меткой `Survived` достаточно объединённого train-набора.

In [ ]:
df = pd.read_csv(DATA_PATH)

# .head() — первые строки, чтобы глазами проверить столбцы и типы значений
display(df.head())

# Общая информация: типы столбцов, число non-null (пропуски видны по разнице с len(df))
print(df.info())

# Описательная статистика для числовых признаков
display(df.describe(include="all").T)

n_rows, n_cols = df.shape
print(f"Размер таблицы: {n_rows} строк × {n_cols} столбцов")
print("Баланс класса Survived:\n", df["Survived"].value_counts(normalize=True).round(3))

## 3. EDA (Exploratory Data Analysis)

**EDA** — исследование данных до моделирования: пропуски, распределения, связь признаков с целевой переменной.

In [ ]:
# --- Пропуски: тепловая карта и столбчатая диаграмма ---
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(df.isna(), cbar=False, yticklabels=False, ax=axes[0])
axes[0].set_title("Карта пропусков (белый = NaN)")

if len(missing):
    sns.barplot(x=missing.values, y=missing.index, ax=axes[1])
    axes[1].set_title("Число пропусков по столбцам")
    axes[1].set_xlabel("Количество")
else:
    axes[1].text(0.5, 0.5, "Пропусков нет", ha="center", va="center")
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()

print("Столбцы с пропусками:")
print(missing if len(missing) else "— нет —")

In [ ]:
# --- Целевая переменная Survived ---
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x="Survived", ax=ax)
ax.set_title("Распределение выживших (0/1)")
ax.set_xticklabels(["Погиб (0)", "Выжил (1)"])
plt.show()

In [ ]:
# --- Числовые признаки: Age, Fare vs Survived ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df, x="Age", hue="Survived", kde=True, element="step", ax=axes[0])
axes[0].set_title("Возраст по классам выживания")

sns.boxplot(data=df, x="Survived", y="Fare", ax=axes[1])
axes[1].set_title("Стоимость билета vs выживание")

plt.tight_layout()
plt.show()

In [ ]:
# --- Категориальные признаки ---
cat_cols = ["Pclass", "Sex", "Embarked", "SibSp", "Parch"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for i, col in enumerate(cat_cols):
    tmp = df.dropna(subset=[col])
    sns.countplot(data=tmp, x=col, hue="Survived", ax=axes[i])
    axes[i].set_title(f"{col} vs Survived")

axes[-1].set_axis_off()
plt.suptitle("Категориальные и дискретные признаки", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Корреляция числовых признаков с Survived ---
num_for_corr = df.select_dtypes(include=[np.number])
corr = num_for_corr.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Матрица корреляций (числовые столбцы)")
plt.show()

### Выводы EDA (кратко)

- **Пол (`Sex`)** и **класс каюты (`Pclass`)** сильно связаны с выживанием.
- **Возраст** и **Fare** дают дополнительный сигнал (дети и пассажиры 1-го класса чаще выживали).
- Много пропусков в **Age**, **Cabin**, **Embarked** — нужен препроцессинг.
- Для обучения уберём идентификаторы и текстовые поля с высокой кардинальностью (`Name`, `Ticket`, `Cabin` в сыром виде).

## 4. Подготовка признаков и разбиение

**Pipeline** гарантирует, что на тестовых данных применяются те же шаги (impute → encode → scale), что и на обучении — без утечки информации.

In [ ]:
# Целевая переменная y и матрица признаков X
TARGET = "Survived"
DROP_COLS = ["PassengerId", "Name", "Ticket", "Cabin"]  # не несут прямой структурированной информации без feature engineering

work = df.drop(columns=DROP_COLS, errors="ignore")
work = work.dropna(subset=[TARGET])

X = work.drop(columns=[TARGET])
y = work[TARGET].astype(int)

numeric_features = ["Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Pclass", "Sex", "Embarked"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),  # медиана устойчива к выбросам
        ("scaler", StandardScaler()),  # важно для LogisticRegression и KNN
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ),  # категории -> бинарные столбцы
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Стратификация сохраняет долю классов в train и test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape)

## 5. Обучение моделей

Обучаем шесть классификаторов в едином формате `Pipeline(preprocessor → model)`.

| Модель | Суть |
|--------|------|
| **Logistic Regression** | Линейная граница в пространстве признаков (в задании — «Linear»). |
| **Decision Tree** | Правила вида «если Age < 12 и Sex=female → …». |
| **Random Forest** | Много деревьев, усреднение голосов. |
| **XGBoost** | Градиентный бустинг по деревьям, оптимизация по loss. |
| **CatBoost** | Бустинг с нативной поддержкой категорий (у нас уже OHE). |
| **KNN** | Класс по k ближайшим соседям в масштабированном пространстве. |

In [ ]:
def make_pipeline(model):
    """Собираем полный конвейер: препроцессинг + оценщик."""
    return Pipeline(steps=[("prep", preprocessor), ("model", model)])


models = {
    "LogisticRegression": make_pipeline(
        LogisticRegression(max_iter=2000, random_state=42)
    ),
    "DecisionTree": make_pipeline(
        DecisionTreeClassifier(max_depth=6, random_state=42)
    ),
    "RandomForest": make_pipeline(
        RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
    ),
    "XGBoost": make_pipeline(
        XGBClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1,
        )
    ),
    "CatBoost": make_pipeline(
        CatBoostClassifier(
            iterations=300,
            depth=4,
            learning_rate=0.05,
            verbose=0,
            random_state=42,
        )
    ),
    "KNN": make_pipeline(
        KNeighborsClassifier(n_neighbors=15)
    ),
}

results = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    # predict_proba есть у всех перечисленных моделей
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_proba),
        }
    )

metrics_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False)
display(metrics_df.round(4))

In [ ]:
# Подробный отчёт для лучшей модели по ROC-AUC
best_name = metrics_df.iloc[0]["model"]
best_pipe = models[best_name]
y_pred_best = best_pipe.predict(X_test)

print(f"Лучшая модель по ROC-AUC: {best_name}\n")
print(classification_report(y_test, y_pred_best, target_names=["Погиб", "Выжил"]))

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Погиб", "Выжил"], yticklabels=["Погиб", "Выжил"])
plt.xlabel("Предсказание")
plt.ylabel("Факт")
plt.title(f"Матрица ошибок — {best_name}")
plt.show()

In [ ]:
# Сравнение ROC-AUC на столбчатой диаграмме
plt.figure(figsize=(8, 4))
sns.barplot(data=metrics_df, x="model", y="roc_auc")
plt.xticks(rotation=25, ha="right")
plt.title("Сравнение моделей (ROC-AUC на тесте)")
plt.ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

## 6. SHAP — анализ признаков

**SHAP** (SHapley Additive exPlanations) показывает, как каждый признак **увеличивает или уменьшает** вероятность выживания для конкретного пассажира.

Для линейной модели используем `shap.LinearExplainer`; для древесных/бустингов — `shap.TreeExplainer` (быстро и точно для деревьев).

In [ ]:
# Преобразуем train/test в матрицу признаков после OHE (только препроцессор)
prep_fitted = preprocessor.fit(X_train, y_train)
X_train_t = prep_fitted.transform(X_train)
X_test_t = prep_fitted.transform(X_test)

# Имена столбцов после One-Hot
ohe = prep_fitted.named_transformers_["cat"].named_steps["onehot"]
cat_names = list(ohe.get_feature_names_out(categorical_features))
feature_names = numeric_features + cat_names

print(f"Число признаков после кодирования: {len(feature_names)}")

In [ ]:
# --- SHAP для лучшей древесной модели (TreeExplainer) ---
# Если лучшая — LogisticRegression или KNN, ниже есть альтернативный блок

tree_candidates = {"DecisionTree", "RandomForest", "XGBoost", "CatBoost"}
shap_model_name = best_name if best_name in tree_candidates else "RandomForest"
shap_pipe = models[shap_model_name]
shap_estimator = shap_pipe.named_steps["model"]

# Обучаем препроцессор и модель заново в явном виде для SHAP
prep_fitted = shap_pipe.named_steps["prep"]
prep_fitted.fit(X_train, y_train)
X_train_t = prep_fitted.transform(X_train)
X_test_t = prep_fitted.transform(X_test)
shap_estimator.fit(X_train_t, y_train)

explainer = shap.TreeExplainer(shap_estimator)
shap_values = explainer.shap_values(X_test_t)

# Для бинарной классификации у некоторых библиотек shap_values — список [класс0, класс1]
if isinstance(shap_values, list):
    shap_vals_survived = shap_values[1]
else:
    shap_vals_survived = shap_values

print(f"SHAP рассчитан для модели: {shap_model_name}")

In [ ]:
# Summary plot: важность и направление влияния признаков
plt.figure()
shap.summary_plot(
    shap_vals_survived,
    X_test_t,
    feature_names=feature_names,
    show=False,
    max_display=15,
)
plt.title(f"SHAP summary — {shap_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
# Bar plot: средняя |SHAP| — глобальная важность
plt.figure()
shap.summary_plot(
    shap_vals_survived,
    X_test_t,
    feature_names=feature_names,
    plot_type="bar",
    show=False,
    max_display=15,
)
plt.title(f"Средняя важность признаков (|SHAP|)")
plt.tight_layout()
plt.show()

In [ ]:
# Локальное объяснение для одного пассажира из теста (waterfall)
row_idx = 0
explanation = shap.Explanation(
    values=shap_vals_survived[row_idx],
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
    data=X_test_t[row_idx],
    feature_names=feature_names,
)

plt.figure()
shap.plots.waterfall(explanation, max_display=12, show=False)
plt.title(f"SHAP waterfall — тестовый объект #{row_idx}")
plt.tight_layout()
plt.show()

In [ ]:
# Дополнительно: SHAP для логистической регрессии (линейная модель из задания)
log_reg_pipe = models["LogisticRegression"]
log_reg_pipe.fit(X_train, y_train)
prep_lr = log_reg_pipe.named_steps["prep"]
X_train_lr = prep_lr.transform(X_train)
X_test_lr = prep_lr.transform(X_test)

lr_model = log_reg_pipe.named_steps["model"]
lr_model.fit(X_train_lr, y_train)

# masker сообщает SHAP распределение фоновых данных
masker = shap.maskers.Independent(data=X_train_lr)
lr_explainer = shap.LinearExplainer(lr_model, masker)
lr_shap = lr_explainer.shap_values(X_test_lr)

plt.figure()
shap.summary_plot(lr_shap, X_test_lr, feature_names=feature_names, show=False, max_display=12)
plt.title("SHAP summary — LogisticRegression (Linear)")
plt.tight_layout()
plt.show()

## 7. Итог

1. Данные Titanic загружены, проведён EDA с визуализацией пропусков, распределений и связей с `Survived`.
2. Построен воспроизводимый `Pipeline` с импутацией, масштабированием и One-Hot Encoding.
3. Сравнены шесть моделей; метрики: Accuracy, Precision, Recall, F1, ROC-AUC.
4. SHAP показал глобальную и локальную значимость признаков (пол, класс, стоимость билета, возраст и др.).

**Источник данных:** файл совместим с [Titanic на Kaggle](https://www.kaggle.com/c/titanic/data); локальная копия — `titanic.csv` в этой папке (зеркало с открытого репозитория; при работе в соревновании скачайте официальный `train.csv` с Kaggle).